# 00 Convert the Kondor ledger to Parquet

Converts the six files of the Kondor 2020 release in `data/raw/kondor/` to Parquet in `data/processed/`.
The files are without a header. The three dictionaries are gzip-compressed and are read by DuckDB directly.
The three transaction tables are xz-compressed and are streamed through PyArrow in 64 MB blocks.

Expected row counts: 609,963,452 addresses, and about 1.22 and 1.33 billion transaction inputs and outputs.

In [1]:
%pip install --quiet duckdb pyarrow pandas numpy scikit-learn matplotlib

import sys, importlib
print(sys.executable)
print('python', sys.version.split()[0])
for name in ['duckdb', 'pyarrow', 'pandas', 'numpy', 'sklearn', 'matplotlib']:
    print(name, importlib.import_module(name).__version__)

Note: you may need to restart the kernel to use updated packages.
/gpfs/home4/rliang/dataset_paper/.venv/bin/python
python 3.13.15
duckdb 2.0.0
pyarrow 26.0.0
pandas 3.1.0
numpy 2.5.4
sklearn 1.9.1
matplotlib 3.11.2


In [2]:
from pathlib import Path
import duckdb

ROOT      = Path.cwd().parent
RAW       = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
OUTPUTS   = ROOT / 'outputs'

THREADS      = 192
MEMORY_LIMIT = '320GB'
TEMP_DIR     = PROCESSED / 'duckdb_tmp'

con = duckdb.connect()
con.execute(f'PRAGMA threads={THREADS}')
con.execute(f"PRAGMA memory_limit='{MEMORY_LIMIT}'")
con.execute(f"PRAGMA temp_directory='{TEMP_DIR}'")

import lzma
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq

KONDOR = RAW / 'kondor'
PROCESSED.mkdir(parents=True, exist_ok=True)

## Schemas

Column names and types follow the documentation of the Kondor release. Amounts are in satoshis.

In [3]:
GZ_TABLES = {
    'addresses': ('bitcoin_2020_addresses.dat.gz', {'addrID': 'BIGINT', 'address': 'VARCHAR'}),
    'txh':       ('bitcoin_2020_txh.dat.gz',       {'txID': 'BIGINT', 'tx_hash': 'VARCHAR'}),
    'bh':        ('bitcoin_2020_bh.dat.gz',        {'blockID': 'INTEGER', 'hash': 'VARCHAR',
                                                    'block_timestamp': 'BIGINT', 'n_txs': 'INTEGER'}),
}
XZ_TABLES = {
    'tx':    ('bitcoin_2020_tx.dat.xz',    {'txID': pa.int64(), 'blockID': pa.int32(),
                                            'n_inputs': pa.int32(), 'n_outputs': pa.int32()}),
    'txin':  ('bitcoin_2020_txin.dat.xz',  {'txID': pa.int64(), 'input_seq': pa.int32(),
                                            'prev_txID': pa.int64(), 'prev_output_seq': pa.int32(),
                                            'addrID': pa.int64(), 'sum': pa.int64()}),
    'txout': ('bitcoin_2020_txout.dat.xz', {'txID': pa.int64(), 'output_seq': pa.int32(),
                                            'addrID': pa.int64(), 'sum': pa.int64()}),
}

## Conversion

In [4]:
def convert_gz(src, dst, columns):
    select = ', '.join(f'column{i} AS {name}' for i, name in enumerate(columns))
    types = '{' + ', '.join(f"'column{i}': '{t}'" for i, t in enumerate(columns.values())) + '}'
    con.execute(f"""
        COPY (SELECT {select}
              FROM read_csv('{src}', delim='\\t', header=false, columns={types}, compression='gzip'))
        TO '{dst}' (FORMAT parquet, COMPRESSION snappy)""")


def convert_xz(src, dst, columns):
    read = pacsv.ReadOptions(column_names=list(columns), block_size=64 * 1024 * 1024)
    parse = pacsv.ParseOptions(delimiter='\t')
    convert = pacsv.ConvertOptions(column_types=dict(columns))
    writer = None
    with lzma.open(src, 'rb') as fh:
        for batch in pacsv.open_csv(fh, read_options=read, parse_options=parse, convert_options=convert):
            if writer is None:
                writer = pq.ParquetWriter(dst, batch.schema, compression='snappy')
            writer.write_batch(batch)
    writer.close()


for name, (file, columns) in GZ_TABLES.items():
    convert_gz(KONDOR / file, PROCESSED / f'kondor_{name}.parquet', columns)
for name, (file, columns) in XZ_TABLES.items():
    convert_xz(KONDOR / file, PROCESSED / f'kondor_{name}.parquet', columns)

## Row counts

In [5]:
for name in [*GZ_TABLES, *XZ_TABLES]:
    n = con.execute(f"SELECT COUNT(*) FROM '{PROCESSED / f'kondor_{name}.parquet'}'").fetchone()[0]
    print(f'kondor_{name:10s} {n:>15,}')

kondor_addresses      609,963,452
kondor_txh            500,663,153
kondor_bh                 616,346
kondor_tx             500,663,153
kondor_txin         1,224,170,162
kondor_txout        1,330,388,050
